<a href="https://colab.research.google.com/github/rfaoktvian/Credit_Risk_Prediction_ID-X-Partners/blob/main/CreditRisk_Final_Project_ID_X_Partners.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### BUSINESS UNDERSTANDING

### IMPORT LIBRARY

In [191]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### LOAD DATA

In [192]:
path = '/content/drive/My Drive/Kuliah/PortfolioProject/IDX Partners Final Project/loan_data_2007_2014.csv'
#path = 'loan_data_2007_2014.csv'
df = pd.read_csv(path)

KeyboardInterrupt: 

### DATA UNDERSTANDING

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
pd.set_option('display.max_columns', None)


In [ ]:
df.head()

### FEATURE ENGINEERING

In [ ]:
# Hapus kolom yang tidak digunakan
cols_to_drop = [

# unique id
'Unnamed: 0',
'id',
'member_id',

# free text
'url',
'desc',

# all null / constant / others
'zip_code',
'annual_inc_joint',
'dti_joint',
'verification_status_joint',
'open_acc_6m',
'open_il_6m',
'open_il_12m',
'open_il_24m',
'mths_since_rcnt_il',
'total_bal_il',
'il_util',
'open_rv_12m',
'open_rv_24m',
'max_bal_bc',
'all_util',
'inq_fi',
'total_cu_tl',
'inq_last_12m',
'mths_since_last_major_derog',
'tot_coll_amt',
'tot_cur_bal',
'total_rev_hi_lim',

# leakage cols
 'out_prncp', 'out_prncp_inv',
  'total_pymnt', 'total_pymnt_inv',
  'total_rec_prncp', 'total_rec_int',
  'total_rec_late_fee',
  'recoveries', 'collection_recovery_fee',
  'last_pymnt_amnt',
  'mths_since_last_pymnt_d',
  'mths_since_next_pymnt_d',
  'mths_since_last_credit_pull_d'


# expert judgment
'sub_grade'

]

In [ ]:
df.drop(cols_to_drop, axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Defining Target Feature
persentase_loan_status = df['loan_status'].value_counts(normalize=True)
print(persentase_loan_status)

In [ ]:
# Klasifikasi Status

bad= ['Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off', 'In Grace Period']

df['bad_status'] = np.where(df['loan_status'].isin(bad),1,0)


In [ ]:
persentase_bad_status = df['bad_status'].value_counts(normalize=True)*100

print(persentase_bad_status)


In [ ]:
# Hapus fitur asli 'loan status'

df.drop('loan_status', axis=1, inplace=True)

### DATA CLEANING

In [ ]:
df.head()

In [ ]:
# 1. emp_length
df['emp_length'].unique()

Nilai dari fitur emp_length ini memiliki kaya 'years' dan karakter tambahan yang tidak perlu, jadi perlu dihapus.

In [ ]:
df['emp_length'] = df['emp_length'].str.replace(r'[^0-9]+','',regex=True).astype(float)


In [ ]:
df['emp_length'].unique()

In [ ]:
# 2. term
df['term'].unique()

In [ ]:
df['term'] = df['term'].str.replace('months','').astype(float)

In [ ]:
df['term'].unique()

In [ ]:
# 3. earliest_cr_line
df['earliest_cr_line'].head(5)


Perlu mengubah format earliest_cr_line tersebut menjadi dormat tanggal / date.

In [ ]:
df['earliest_cr_line_date'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%y')
df['earliest_cr_line_date'].head(5)

In [ ]:
# menghitung jumlah bulan sejak 'earliest_cr_line' hingga tanggal referensi
df['mths_since_earliest_cr_line'] = round((pd.to_datetime('2017-12-01') - df['earliest_cr_line_date']) / pd.Timedelta(days=30.44))
df['mths_since_earliest_cr_line'].head(5)


In [ ]:
df['mths_since_earliest_cr_line'].describe()

Terdapat anomali ditemukan pada minimum months since earliest credit line, karena valuenya seharusnya tidak bisa minus. Karena mnths since earliest credit line ini data referensi (2017) yang ada di masa depan dikurangi dengan pembuatan credit line di masa lalu, jadi tidak mungkin masa lalu lebih besar dibanding masa refernsi / terakhir data diambil

In [ ]:
df[df['mths_since_earliest_cr_line'] < 0][['earliest_cr_line','earliest_cr_line_date','mths_since_earliest_cr_line']].head(3)

Terdapat temuan kesalahan konversi tahun dari tahun 1962 menjadi 2026, hal ini yang menyebabkan months since earliest credit line nya memiliki nilai minus.

Perlu dilakukan preprocessing untuk memperbaiki konversi tersebut agar sesuai, jadi solusi yang dapat diterapkan yaitu mengubah nilai yang negatif, menjadi nilai maximum dari fitur tersebut. Karena mengetahui nilai-nilai yang negatif ini merupakan sebuah outlier dari dataset 2007 - 2014, yang artinya sudah sangat tua (1900an), jadi masih masuk akal jika dijadikan nilai maksimal.

In [ ]:
# Mengubah menjadi nilai max
df.loc[df['mths_since_earliest_cr_line'] < 0, 'mths_since_earliest_cr_line'] = df['mths_since_earliest_cr_line'].max()

In [ ]:
# Drop kolom
df.drop(['earliest_cr_line','earliest_cr_line_date'], axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
# 4. issue_d

df['issue_d'].head(3)

Pada fitur issu_d / issue date / tanggal pencairan ini masih memiliki tipe data 'object', perlu adanya konversi menjadi tipe data date.

In [ ]:
# konversi tipe data ke date
df['issue_d_date'] = pd.to_datetime(df['issue_d'], format = '%b-%y')

df['issue_d_date'].head(3)

In [ ]:
# Menghitung jumlah bulan issue date hingga tanggal referensi (2017-12-01)

df['mths_since_issue_d'] = round((pd.to_datetime('2017-12-01') - df['issue_d_date']) / pd.Timedelta(days=30.44) )

In [ ]:
df['mths_since_issue_d'].describe()

In [ ]:
# drop kolom
df.drop(['issue_d','issue_d_date'], axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
# 5. last_payment_d
df['last_pymnt_d'].head(3)

In [ ]:
df['last_pymnt_date'] = pd.to_datetime(df['last_pymnt_d'], format= '%b-%y')
df['last_pymnt_date'].head(3)

In [ ]:
df['mths_since_last_pymnt_d'] = round((pd.to_datetime('2017-12-01') - df['last_pymnt_date']) / pd.Timedelta(days=30.44))

In [ ]:
df['mths_since_last_pymnt_d'].head()

In [ ]:
df['mths_since_last_pymnt_d'].describe()

In [ ]:
df.drop(['last_pymnt_d','last_pymnt_date'],axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
# 6.next_pymnt_d
df['next_pymnt_d'].unique()

Fitur ini sama dengan fitur sebelumnya, hanya menampilkan tahun dan bulan, perlu dikonversi ke bentuk date agar dapat di analisis

In [ ]:
df['next_pymnt_date'] = pd.to_datetime(df['next_pymnt_d'], format= '%b-%y')
df['next_pymnt_date'].unique()

In [ ]:
df['mths_since_next_pymnt_d'] = round((pd.to_datetime('2017-12-01')- df['next_pymnt_date']) / pd.Timedelta(days=30.44))
df['mths_since_next_pymnt_d'].unique()

In [ ]:
df['mths_since_next_pymnt_d'].describe()

In [ ]:
df.drop(['next_pymnt_d','next_pymnt_date'], axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
# 7. last_credit_pull_d

df['last_credit_pull_d'].head()

In [ ]:
df['last_credit_pull_date'] = pd.to_datetime(df['last_credit_pull_d'], format= '%b-%y')
df['last_credit_pull_date'].head()

In [ ]:
df['mths_since_last_credit_pull_d'] = round((pd.to_datetime('2017-12-01') - df['last_credit_pull_date']) / pd.Timedelta(days=30.44))
df['mths_since_last_credit_pull_d'].head()

In [ ]:
df['mths_since_last_credit_pull_d'].describe()

In [ ]:
df.drop(['last_credit_pull_d', 'last_credit_pull_date'], axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
df.dtypes

#### EXPXLORATORY DATA ANALYSIS

In [ ]:
numerical_df = df.select_dtypes(include=['int64', 'float64'])
categorical_df = df.select_dtypes(include='object')

In [ ]:
# Correlation Matrix

corr_matrix = numerical_df.corr()
plt.figure(figsize=(16, 12))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    vmin=-1, vmax=1,
    center=0
)
plt.title('Correlation Matrix of Numerical Features', fontsize=16)
plt.show()


#### Check Cardinality Data

In [ ]:
categorical_df.nunique()

In [ ]:
numerical_df.nunique()

In [ ]:
df.drop(['policy_code'],axis=1, inplace=True)
df.shape

In [ ]:
for col in categorical_df.columns.tolist():
    print('Distribusi nilai untuk fitur',col)
    print(df[col].value_counts(normalize=True) *100)


Nilai distribusi pada fitur pymnt_plan sagat bias, 99% termasuk ke nilai n dan sisanya y, maka dari itu fitur ini perlu dihapus agar tidak menyebabkan bias pada model

In [ ]:
df.drop(['pymnt_plan'],axis=1, inplace=True)
df.shape

### UNIVARIATE ANALYSIS

In [ ]:
# Cek kolom categorical yang tersisa
print(df.select_dtypes(include='object').columns.tolist())
print(f"Jumlah kolom categorical: {len(df.select_dtypes(include='object').columns)}")

In [ ]:
categorical_df.nunique()

Fitur emp_title dan title memiliki banyak sekali nilai unik, perlu dihapus agar visualisai distribusi categorical data dapat berjalan

In [ ]:
df.drop(['emp_title', 'title', 'application_type'], axis=1, inplace=True)


In [ ]:
categorical_df = df.select_dtypes(include=['object']).columns
categorical_df

In [ ]:
# Distribusi Categorical Data
plt.style.use("ggplot")

for column in categorical_df:
    plt.figure(figsize=(20, 4))
    plt.subplot(121)
    df[column].value_counts().plot(kind="bar")
    plt.xlabel(column)
    plt.ylabel("number of customers")
    plt.title(column)
    plt.tight_layout()
    plt.show()

In [ ]:
numerical_df = df.select_dtypes(include=np.number)
numerical_df

In [ ]:
numerical_df.head()

In [ ]:
plt.style.use('ggplot')
for column in numerical_df.columns.tolist():
  average = df[column].mean()
  median = df[column].median()
  mode = df[column].mode().iloc[0]
  std = df[column].std()


  plt.figure(figsize=(20,4))
  plt.subplot(121)
  sns.displot(df[column], kde=True)
  plt.axvline(average, color='red', linestyle='dashed', linewidth=3, label='mean')
  plt.axvline(median, color='green', linestyle='dashed', linewidth=3, label='median')
  plt.axvline(mode, color='blue', linestyle='dashed', linewidth=3, label='mode')
  plt.title(column)

  print('Ringkasan statistik {columns}'.format(columns=column))
  print('Rata-rata: {average}'.format(average=average))
  print('Median: {median}'.format(median=median))
  print('Modus: {mode}'.format(mode=mode))
  print('Standar Deviasi: {std}'.format(std=std))


### BIVARIATE ANALYSIS

In [ ]:
plt.style.use('ggplot')
for column in categorical_df:
  plt.figure(figsize=(20,4))
  plt.subplot(121)
  sns.countplot(x=column, hue='bad_status', data=df)
  plt.title(column)
  plt.xticks(rotation=90)

In [ ]:
missing_values_pct = df.isnull().sum() * 100 / df.shape[0]

missing_values = missing_values_pct[missing_values_pct > 0].sort_values(ascending=False)
print('Presentase nilai yang hilang untuk setiap fitur:')
print(missing_values)

Ditemukan missing values lebih dari 80%, yang artinya sangat banyak dan tidak bisa digunakan, sehaingga perlu di drop

In [ ]:
df.drop('mths_since_last_record',axis=1, inplace=True)

#### Data Imputation

In [ ]:
df.columns

In [ ]:
df['annual_inc'].fillna(df['annual_inc'].median(), inplace=True)
df['acc_now_delinq'].fillna(0, inplace=True)
df['total_acc'].fillna(0, inplace=True)
df['pub_rec' ]. fillna(0, inplace=True)
df['open_acc'].fillna(0, inplace=True)
df['inq_last_6mths']. fillna(0, inplace=True)
df['delinq_2yrs'].fillna(0, inplace=True)
df['collections_12_mths_ex_med'].fillna(0, inplace=True)
df['revol_util']. fillna(0, inplace=True)
df['emp_length'].fillna(0, inplace=True)
df['mths_since_last_delinq']. fillna(-1, inplace=True)
df['mths_since_next_pymnt_d'].fillna(-1, inplace=True)
df['mths_since_last_pymnt_d'].fillna(df['mths_since_last_pymnt_d'].median(), inplace=True)
df['mths_since_last_credit_pull_d'].fillna(df['mths_since_last_credit_pull_d'].median(), inplace=True)
df['mths_since_earliest_cr_line'].fillna(df['mths_since_earliest_cr_line'].median(), inplace=True)


### LABEL ENCODING

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for column in categorical_df:
  df[column] = le.fit_transform(df[column])

df.head()

### DEFINE TARGET



In [ ]:
X = df.drop('bad_status', axis=1)
y = df['bad_status']

In [ ]:
y.value_counts()

Ditemukan imbalaced data untuk fitur target, sehingga perlu dilakukan preprocessing untuk menangani imbalance data ini menggunakan SMOTE

In [ ]:
X.isnull().sum()

#### Train Split & Test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape
X_test.shape

#### Handling Imbalance Data

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, sampling_strategy=1)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [ ]:
y.value_counts()

Berhasil melakukan SMOTE

### DATA PREPROCESSING

#### Standarization

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train .columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [ ]:
X_train_scaled.head()

### DATA MODELLING

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(random_state=42)
logreg.fit(X_train_scaled, y_train_res)



In [ ]:
y_train_pred = logreg.predict(X_train_scaled)
y_test_pred = logreg.predict(X_test_scaled)

### EVALUATION

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, classification_report

y_prob = logreg.predict_proba(X_test_scaled)[:, 1]
y_pred = logreg.predict(X_test_scaled)

roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC: {roc_auc:.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
X.columns.tolist()